# Colab: M3 (fine-tuning de BETO) con checkpoints en Google Drive

Entrena M3 con las 3 semillas sobre los splits de SAIJ y compara contra M0, M1 y M2 (que ya se corrieron en el Mac).

**Cómo no perder el avance si Colab se apaga.** Todo lo que importa vive en Drive, no en la sesión de Colab:

- M3 guarda un **checkpoint al terminar cada época** (modelo, optimizador, historial y azar) en `Drive/proyecto_final_dl/artifacts/ckpt/`.
- Cada corrida terminada deja sus predicciones y su `run.json` en `Drive/proyecto_final_dl/results/`.
- Si se corta: **Entorno de ejecución > Reiniciar sesión**, vuelve a correr las celdas 1 a 4 y luego la celda de entrenamiento. El mismo comando omite lo ya terminado y continúa M3 desde la última época guardada.

**Antes de empezar (una vez):**

1. En tu Mac: `python -m src.colab_bundle` genera `artifacts/colab_bundle.zip` (código, configuración, splits de train y val, y resultados de M0 a M2; sin test ni datos del bufete).
2. Sube ese zip a Drive, a la carpeta `Mi unidad/proyecto_final_dl/`.
3. Entorno de ejecución > Cambiar tipo de entorno > **GPU (T4)**.

**Ojo:** M0, M1 y M2 salen de la corrida en el Mac y M3 de la GPU de Colab; los tiempos de cada uno no son comparables entre sí.

In [ ]:
# 1. GPU
import torch
print("GPU disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("ATENCIÓN: sin GPU el fine-tuning tarda horas. Entorno de ejecución > Cambiar tipo de entorno > GPU.")

In [ ]:
# 2. Drive, código y rutas persistentes
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib, shutil, zipfile
PROJ = pathlib.Path('/content/drive/MyDrive/proyecto_final_dl')     # carpeta donde subiste colab_bundle.zip
REPO = pathlib.Path('/content/proyecto_final-DL')                   # copia local del código (rápida)

if not REPO.exists():
    with zipfile.ZipFile(PROJ / 'colab_bundle.zip') as z:
        z.extractall('/content')

# Lo que debe sobrevivir a una desconexión vive en Drive:
os.environ['DL_PATH_RESULTS'] = str(PROJ / 'results')
os.environ['DL_PATH_ARTIFACTS'] = str(PROJ / 'artifacts')
for d in ('results', 'artifacts'):
    (PROJ / d).mkdir(parents=True, exist_ok=True)

# resultados de M0 a M2 del Mac -> Drive (sin pisar lo que ya haya)
for f in (REPO / 'experiments' / 'results').glob('*'):
    if not (PROJ / 'results' / f.name).exists():
        shutil.copy(f, PROJ / 'results' / f.name)

%cd /content/proyecto_final-DL
print('Resultados en', os.environ['DL_PATH_RESULTS'])
print('Checkpoints y pesos en', os.environ['DL_PATH_ARTIFACTS'])

In [ ]:
# 3. Dependencias (mismas versiones de transformers y accelerate que el Mac)
!pip -q install "transformers==4.46.3" "accelerate==0.34.2" scikit-learn pandas pyyaml matplotlib

In [ ]:
# 4. Comprobación rápida: el código y los splits están donde deben
import pandas as pd
from src.config import load_config
cfg = load_config()
train, val = pd.read_csv('data/processed/train.csv'), pd.read_csv('data/processed/val.csv')
print('train', len(train), '| val', len(val))
print(train['area_legal'].value_counts().to_dict())
print('M3:', {k: cfg['models']['m3'][k] for k in ('lr', 'batch_size', 'epochs', 'patience', 'max_len')})

## Entrenar

Esta celda hace todo y se puede **volver a lanzar las veces que haga falta**: omite lo terminado y reanuda M3 en la última época guardada. Cada época imprime pérdida, F1 en validación y minutos.

Con una GPU T4 y precisión mixta estimo unos minutos por época sobre 12.600 textos; la primera época dice el tiempo real. Si falla por memoria (`CUDA out of memory`), baja `batch_size` en la celda siguiente y vuelve a correr.

In [ ]:
# (Opcional) bajar el tamaño de lote si hay error de memoria. Cambia el número y ejecuta SOLO si hace falta.
# OJO: si ya hay checkpoints con otro batch_size, bórralos antes (Drive/proyecto_final_dl/artifacts/ckpt) o dará error.
# import yaml
# c = yaml.safe_load(open('configs/config.yaml')); c['models']['m3']['batch_size'] = 8
# yaml.safe_dump(c, open('configs/config.yaml', 'w'), allow_unicode=True, sort_keys=False)

In [ ]:
# 5. ENTRENAR: M0 (segundos), M1 y M2 se omiten (ya vienen del Mac), M3 con 3 semillas
!python -m src.reproduce --stage val --models m0 m1 m2 m3 --seeds 42 43 44

## Ver el avance (sin entrenar)

Útil mientras corre o después de un corte: muestra en qué época va cada semilla y qué corridas ya terminaron.

In [ ]:
import json, glob, os
for st in sorted(glob.glob(os.environ['DL_PATH_ARTIFACTS'] + '/ckpt/*/status.json')):
    s = json.load(open(st))
    print(os.path.basename(os.path.dirname(st)), '->', s)
print()
for f in sorted(glob.glob(os.environ['DL_PATH_RESULTS'] + '/m3*_val_run.json')):
    i = json.load(open(f))
    print('TERMINADO', i['model'], '| mejor época', i['best_epoch'], 'de', i['epochs_run'], '| F1 val', round(i['best_val_macro_f1'], 4),
          '| minutos', round(i['train_seconds'] / 60, 1), '| reanudado desde época', i['resumed_from_epoch'])

## Resultado: M2 contra M3

Delta de F1 macro con intervalo pareado, recall por clase y costos.

In [ ]:
from src.evaluation.compare import compare_files
R = os.environ['DL_PATH_RESULTS']
res, tabla = compare_files(f'{R}/m2_val_preds.csv', f'{R}/m3_val_preds.csv', list(cfg['areas']), 'M2', 'M3')
print(tabla)

In [ ]:
print(open(os.environ['DL_PATH_RESULTS'] + '/summary_val.md', encoding='utf-8').read())

## Llevarse los resultados al repo

Los resultados ya están en Drive (`proyecto_final_dl/results/`). Para copiarlos al repo del Mac, descarga la carpeta o este zip y ponlo en `experiments/results/`. Los pesos de M3 (`m3.pt`, ~440 MB, hacen falta para la API) quedan en `proyecto_final_dl/artifacts/`: cópialos a mano desde Drive si los necesitas.

In [ ]:
!cd "$DL_PATH_RESULTS" && zip -q -r /content/resultados_colab.zip . -x "*.pt"
from google.colab import files
files.download('/content/resultados_colab.zip')